# Advanced Problems with Solutions: Positional, Default, and Keyword Arguments

These problems focus on Python function calls, argument binding rules, default values, keyword arguments, and common edge cases.

## Problem 1: Trace Argument Binding

Given the function below, predict the output of each call before running it.

```python
def describe_user(username, role='member', active=True):
    return f'{username=}, {role=}, {active=}'
```

In [1]:
def describe_user(username, role='member', active=True):
    return f'{username=}, {role=}, {active=}'

print(describe_user('maya'))
print(describe_user('maya', 'admin'))
print(describe_user('maya', active=False))
print(describe_user(role='moderator', username='leo'))

username='maya', role='member', active=True
username='maya', role='admin', active=True
username='maya', role='member', active=False
username='leo', role='moderator', active=True


### Solution 1

```text
username='maya', role='member', active=True
username='maya', role='admin', active=True
username='maya', role='member', active=False
username='leo', role='moderator', active=True
```

Python first assigns positional arguments from left to right. Keyword arguments are then matched by parameter name. Any omitted parameters use their default values.

## Problem 2: Identify Invalid Calls

For the function below, determine which calls are valid and which raise an error.

```python
def order(item, quantity=1, express=False):
    return item, quantity, express
```

In [2]:
def order(item, quantity=1, express=False):
    return item, quantity, express

calls = [
    "order('book')",
    "order('book', 3)",
    "order('book', express=True)",
    "order(item='book', 3)",
    "order('book', quantity=2, express=True)",
    "order('book', item='pen')"
]

for call in calls:
    try:
        print(call, '=>', eval(call))
    except Exception as e:
        print(call, '=>', type(e).__name__, e)

order('book') => ('book', 1, False)
order('book', 3) => ('book', 3, False)
order('book', express=True) => ('book', 1, True)
order(item='book', 3) => SyntaxError positional argument follows keyword argument (<string>, line 1)
order('book', quantity=2, express=True) => ('book', 2, True)
order('book', item='pen') => TypeError order() got multiple values for argument 'item'


### Solution 2

Valid:

```python
order('book')
order('book', 3)
order('book', express=True)
order('book', quantity=2, express=True)
```

Invalid:

```python
order(item='book', 3)
```

This is invalid because a positional argument cannot appear after a keyword argument.

```python
order('book', item='pen')
```

This is invalid because `item` receives two values: one positionally and one by keyword.

## Problem 3: Fix a Broken Function Definition

The function below is invalid. Fix it in two different ways.

```python
def create_profile(name='Anonymous', email, verified=False):
    return name, email, verified
```

In [3]:
# Solution A: Move the required parameter before default parameters
def create_profile_a(email, name='Anonymous', verified=False):
    return name, email, verified

# Solution B: Give every parameter after the first default a default too
def create_profile_b(name='Anonymous', email=None, verified=False):
    return name, email, verified

print(create_profile_a('a@example.com'))
print(create_profile_b(email='b@example.com'))

('Anonymous', 'a@example.com', False)
('Anonymous', 'b@example.com', False)


### Solution 3

A parameter without a default value cannot follow a parameter with a default value.

Best practice is usually Solution A, because `email` appears to be required and should therefore come first.

## Problem 4: Design a Safe API

Write a function `calculate_invoice` that accepts:

- `subtotal`, required
- `tax_rate`, default `0.2`
- `discount`, default `0`
- `shipping`, default `0`

Return the final total after discount, tax, and shipping.

Formula:

```text
final = (subtotal - discount) * (1 + tax_rate) + shipping
```

In [4]:
def calculate_invoice(subtotal, tax_rate=0.2, discount=0, shipping=0):
    if subtotal < 0:
        raise ValueError('subtotal cannot be negative')
    if discount < 0:
        raise ValueError('discount cannot be negative')
    if discount > subtotal:
        raise ValueError('discount cannot exceed subtotal')
    if shipping < 0:
        raise ValueError('shipping cannot be negative')
    return round((subtotal - discount) * (1 + tax_rate) + shipping, 2)

print(calculate_invoice(100))
print(calculate_invoice(100, discount=10))
print(calculate_invoice(100, shipping=5))
print(calculate_invoice(100, tax_rate=0.1, discount=20, shipping=7.5))

120.0
108.0
125.0
95.5


### Solution 4

Expected output:

```text
120.0
108.0
125.0
95.5
```

Using keyword arguments improves readability when several numeric defaults are involved.

## Problem 5: Avoid Ambiguous Positional Calls

The function below works, but calls to it can be hard to read.

```python
def resize(width, height, keep_aspect=True, upscale=False):
    return width, height, keep_aspect, upscale
```

Rewrite the calls below using keyword arguments where appropriate.

```python
resize(800, 600, False, True)
resize(1920, 1080, True, False)
```

In [5]:
def resize(width, height, keep_aspect=True, upscale=False):
    return width, height, keep_aspect, upscale

print(resize(800, 600, keep_aspect=False, upscale=True))
print(resize(1920, 1080, keep_aspect=True, upscale=False))

(800, 600, False, True)
(1920, 1080, True, False)


### Solution 5

Boolean arguments are often unclear when passed positionally.

Prefer this:

```python
resize(800, 600, keep_aspect=False, upscale=True)
```

over this:

```python
resize(800, 600, False, True)
```

## Problem 6: Debug Multiple Values for One Argument

Explain why this code fails and fix it.

```python
def connect(host, port=5432, timeout=30):
    return f'Connecting to {host}:{port} with timeout={timeout}'

connect('localhost', host='db.example.com')
```

In [6]:
def connect(host, port=5432, timeout=30):
    return f'Connecting to {host}:{port} with timeout={timeout}'

# Fixed versions:
print(connect('db.example.com'))
print(connect(host='db.example.com'))
print(connect('localhost', port=5433, timeout=10))

Connecting to db.example.com:5432 with timeout=30
Connecting to db.example.com:5432 with timeout=30
Connecting to localhost:5433 with timeout=10


### Solution 6

`connect('localhost', host='db.example.com')` fails because `host` receives two values:

- `'localhost'` positionally
- `'db.example.com'` by keyword

Each parameter may be assigned only once.

## Problem 7: Mutable Default Argument Trap

Predict the output of this function, then fix it.

```python
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags
```

In [7]:
def add_tag_bad(tag, tags=[]):
    tags.append(tag)
    return tags

print(add_tag_bad('python'))
print(add_tag_bad('data'))
print(add_tag_bad('ml'))

['python']
['python', 'data']
['python', 'data', 'ml']


In [8]:
def add_tag(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

print(add_tag('python'))
print(add_tag('data'))
print(add_tag('ml'))
print(add_tag('advanced', ['python']))

['python']
['data']
['ml']
['python', 'advanced']


### Solution 7

Mutable default values are created once, when the function is defined, not each time the function is called.

Bad output:

```text
['python']
['python', 'data']
['python', 'data', 'ml']
```

Best practice: use `None` as the default and create a new list inside the function.

## Problem 8: Build a Function with Clear Defaults

Create a function `format_name` that accepts:

- `first`, required
- `last`, required
- `middle`, optional, default `None`
- `title`, optional, default empty string

The function should return a neatly formatted full name.

In [9]:
def format_name(first, last, middle=None, title=''):
    parts = []
    if title:
        parts.append(title)
    parts.append(first)
    if middle:
        parts.append(middle)
    parts.append(last)
    return ' '.join(parts)

print(format_name('Ada', 'Lovelace'))
print(format_name('John', 'Smith', middle='Quincy'))
print(format_name('Grace', 'Hopper', title='Dr.'))
print(format_name('Martin', 'King', middle='Luther', title='Dr.'))

Ada Lovelace
John Quincy Smith
Dr. Grace Hopper
Dr. Martin Luther King


### Solution 8

Expected output:

```text
Ada Lovelace
John Quincy Smith
Dr. Grace Hopper
Dr. Martin Luther King
```

Required arguments come first. Optional arguments with defaults come after them.

## Problem 9: Function Call Matrix

For the function below, test each call and classify it as valid or invalid.

```python
def schedule(task, day='Monday', hour=9):
    return f'{task} scheduled on {day} at {hour}:00'
```

In [10]:
def schedule(task, day='Monday', hour=9):
    return f'{task} scheduled on {day} at {hour}:00'

test_calls = [
    "schedule('Backup')",
    "schedule('Backup', 'Friday')",
    "schedule('Backup', hour=23)",
    "schedule(day='Friday', task='Backup')",
    "schedule('Backup', task='Cleanup')",
    "schedule(hour=10, 'Backup')"
]

for call in test_calls:
    try:
        result = eval(call)
        print(f'VALID:   {call} -> {result}')
    except Exception as e:
        print(f'INVALID: {call} -> {type(e).__name__}: {e}')

VALID:   schedule('Backup') -> Backup scheduled on Monday at 9:00
VALID:   schedule('Backup', 'Friday') -> Backup scheduled on Friday at 9:00
VALID:   schedule('Backup', hour=23) -> Backup scheduled on Monday at 23:00
VALID:   schedule(day='Friday', task='Backup') -> Backup scheduled on Friday at 9:00
INVALID: schedule('Backup', task='Cleanup') -> TypeError: schedule() got multiple values for argument 'task'
INVALID: schedule(hour=10, 'Backup') -> SyntaxError: positional argument follows keyword argument (<string>, line 1)


### Solution 9

Valid calls:

```python
schedule('Backup')
schedule('Backup', 'Friday')
schedule('Backup', hour=23)
schedule(day='Friday', task='Backup')
```

Invalid calls:

```python
schedule('Backup', task='Cleanup')
```

`task` receives two values.

```python
schedule(hour=10, 'Backup')
```

A positional argument cannot follow a keyword argument.

## Problem 10: Refactor for Readability

The function call below is technically valid but hard to understand.

```python
send_notification('Server down', 'admin@example.com', True, False, 3)
```

Refactor the function and call to make the API clearer.

In [11]:
def send_notification(message, recipient, urgent=False, retry=False, attempts=1):
    return {
        'message': message,
        'recipient': recipient,
        'urgent': urgent,
        'retry': retry,
        'attempts': attempts
    }

notification = send_notification(
    'Server down',
    'admin@example.com',
    urgent=True,
    retry=False,
    attempts=3
)

print(notification)

{'message': 'Server down', 'recipient': 'admin@example.com', 'urgent': True, 'retry': False, 'attempts': 3}


### Solution 10

Use positional arguments for the most essential values and keyword arguments for optional configuration.

This is clearer:

```python
send_notification(
    'Server down',
    'admin@example.com',
    urgent=True,
    retry=False,
    attempts=3
)
```

than this:

```python
send_notification('Server down', 'admin@example.com', True, False, 3)
```

## Challenge Problem: Mini Argument Binder

Write a simplified function called `bind_arguments` that mimics how Python assigns arguments to parameters.

It should accept:

- `parameters`: list of parameter names
- `defaults`: dictionary of default values
- `args`: tuple of positional arguments
- `kwargs`: dictionary of keyword arguments

It should return a dictionary mapping parameter names to final values.

Raise an error if:

- too many positional arguments are provided
- a parameter receives multiple values
- a required argument is missing
- an unexpected keyword argument is provided

In [12]:
def bind_arguments(parameters, defaults=None, args=(), kwargs=None):
    defaults = defaults or {}
    kwargs = kwargs or {}
    bound = {}

    if len(args) > len(parameters):
        raise TypeError('too many positional arguments')

    for name, value in zip(parameters, args):
        bound[name] = value

    for name, value in kwargs.items():
        if name not in parameters:
            raise TypeError(f'unexpected keyword argument: {name}')
        if name in bound:
            raise TypeError(f'multiple values for argument: {name}')
        bound[name] = value

    for name in parameters:
        if name not in bound:
            if name in defaults:
                bound[name] = defaults[name]
            else:
                raise TypeError(f'missing required argument: {name}')

    return bound


params = ['a', 'b', 'c']
defaults = {'b': 2, 'c': 3}

print(bind_arguments(params, defaults, args=(10,), kwargs={'c': 30}))
print(bind_arguments(params, defaults, args=(10, 20, 30)))

for bad_call in [
    lambda: bind_arguments(params, defaults, args=(1, 2, 3, 4)),
    lambda: bind_arguments(params, defaults, args=(1,), kwargs={'a': 99}),
    lambda: bind_arguments(params, defaults, kwargs={'x': 1}),
    lambda: bind_arguments(params, defaults, args=())
]:
    try:
        print(bad_call())
    except TypeError as e:
        print(type(e).__name__, e)

{'a': 10, 'c': 30, 'b': 2}
{'a': 10, 'b': 20, 'c': 30}
TypeError too many positional arguments
TypeError multiple values for argument: a
TypeError unexpected keyword argument: x
TypeError missing required argument: a


### Challenge Solution

This problem models Python's argument binding rules:

1. Positional arguments bind first.
2. Keyword arguments bind by name.
3. No parameter may receive more than one value.
4. Missing parameters use defaults when available.
5. Required parameters without values raise an error.

This is the same reasoning behind errors such as:

```text
TypeError: multiple values for argument
TypeError: missing required positional argument
SyntaxError: positional argument follows keyword argument
```